# 02 - Data cleaning and validation

Purpose: turn a raw observation table into a validated, gap-free daily series.

Cleaning policy, implemented in `rainml.data.cleaning`:

* remove physically impossible values (negative, above 1000 mm/day);
* keep real extremes - flag them, do not delete them;
* treat a missing day as 0 mm rather than imputing the mean, because a gap in a
  daily gauge series means no measurable rain, and imputing upward would bias every
  rolling window.

Validation, in `rainml.data.validation`, raises rather than warns: training on an
invalid dataset produces a model card with meaningless metrics.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
from rainml.data.loaders import load_districts, load_observations, attach_coordinates
from rainml.data.cleaning import clean_observations
from rainml.data.validation import validate_observations

observations = attach_coordinates(
    load_observations(OBSERVATIONS), load_districts(DISTRICTS)
)
report = validate_observations(observations)
print("valid:", report.ok)
print("warnings:", report.warnings)
print("errors:", report.errors)
report.summary

In [ ]:
cleaned, cleaning_report = clean_observations(observations)
cleaning_report

## What changed

`flagged_outliers` counts statistical extremes that were kept. Deleting them would
remove precisely the heavy-rain days the model is meant to learn.

In [ ]:
comparison = pd.DataFrame(
    {
        "raw_mean": [observations["rainfall_mm"].mean()],
        "cleaned_mean": [cleaned["rainfall_mm"].mean()],
        "raw_p95": [observations["rainfall_mm"].quantile(0.95)],
        "cleaned_p95": [cleaned["rainfall_mm"].quantile(0.95)],
        "raw_max": [observations["rainfall_mm"].max()],
        "cleaned_max": [cleaned["rainfall_mm"].max()],
    }
)
comparison.round(3)

## Continuity after cleaning

Every district should now have one row per calendar day across the same span.

In [ ]:
cleaned_dates = cleaned.copy()
cleaned_dates["observed_date"] = pd.to_datetime(cleaned_dates["observed_at"]).dt.tz_localize(None).dt.normalize()
continuity = cleaned_dates.groupby("district_id")["observed_date"].agg(["min", "max", "nunique"])
continuity["span_days"] = (continuity["max"] - continuity["min"]).dt.days + 1
continuity["complete"] = continuity["nunique"] == continuity["span_days"]
continuity

In [ ]:
districts_daily = cleaned_dates["district_id"].nunique()
days_daily = cleaned_dates["observed_date"].nunique()
print(f"{districts_daily} districts x {days_daily} days = {districts_daily * days_daily} rows")
print(f"actual rows: {len(cleaned_dates)}")

## Persist the cleaned series

Notebook 04 reads the cleaned file rather than recomputing, so the training run is
reproducible from the command line too.

In [ ]:
cleaned_path = DATA_DIR / "observations_clean.csv"
cleaned.to_csv(cleaned_path, index=False)
print("wrote", cleaned_path, len(cleaned), "rows")